# 🕐 Notebook 7: Context Features — Temporal & Demographics

**Mục tiêu:**
- Phân tích ảnh hưởng của **thời gian** (timestamp) lên rating
- Phân tích ảnh hưởng của **demographics** (gender, age, occupation)
- Genre preferences theo nhóm đối tượng

**Context Features giúp gì?**
- Hiểu hành vi user theo thời gian (mùa, ngày, giờ)
- Hiểu preference khác nhau giữa nhóm tuổi, giới tính
- Cải thiện recommendation bằng context

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')

# Load data
ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')
users   = pd.read_csv('data/processed/users_clean.csv')

print(f'✅ Loaded: {len(ratings):,} ratings, {len(users):,} users')

## A. TEMPORAL ANALYSIS

### A1. Trích xuất Temporal Features

In [ ]:
# Convert timestamp → datetime
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year']      = ratings['datetime'].dt.year
ratings['month']     = ratings['datetime'].dt.month
ratings['dayofweek'] = ratings['datetime'].dt.dayofweek
ratings['hour']      = ratings['datetime'].dt.hour

print('Dataset span:')
print(f'  Từ: {ratings["datetime"].min()}')
print(f'  Đến: {ratings["datetime"].max()}')

ratings[['userId','movieId','rating','timestamp','datetime','year','month','dayofweek','hour']].head()

### A2. Rating theo Năm

In [ ]:
year_stats = ratings.groupby('year')['rating'].agg(['mean', 'count', 'std']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Rating trung bình theo năm
axes[0].plot(year_stats['year'], year_stats['mean'], marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Rating trung bình theo năm')
axes[0].set_xlabel('Năm')
axes[0].set_ylabel('Avg Rating')
axes[0].grid(True, alpha=0.3)

# Số ratings theo năm
axes[1].bar(year_stats['year'], year_stats['count'], color='coral', edgecolor='black')
axes[1].set_title('Số Ratings theo năm')
axes[1].set_xlabel('Năm')
axes[1].set_ylabel('Số ratings')

plt.tight_layout()
plt.savefig('results/charts/07_temporal_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nYear stats:')
print(year_stats)

### A3. Rating theo Ngày trong tuần

In [ ]:
day_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
ratings['day_name'] = ratings['dayofweek'].map(day_names)

day_stats = ratings.groupby('dayofweek')['rating'].agg(['mean', 'count']).reset_index()
day_stats['day_name'] = day_stats['dayofweek'].map(day_names)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(day_stats['day_name'], day_stats['mean'], 
              color=['steelblue']*5 + ['coral','coral'], edgecolor='black')
ax.set_title('Rating trung bình theo ngày trong tuần')
ax.set_xlabel('Ngày')
ax.set_ylabel('Avg Rating')
ax.set_ylim(3.5, 4.0)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, day_stats['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('results/charts/07_temporal_by_dayofweek.png', dpi=150, bbox_inches='tight')
plt.show()

### A4. Rating theo Giờ trong ngày

In [ ]:
hour_stats = ratings.groupby('hour')['rating'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hour_stats['hour'], hour_stats['mean'], color='steelblue', edgecolor='black')
ax.set_title('Rating trung bình theo giờ trong ngày')
ax.set_xlabel('Giờ (0–23)')
ax.set_ylabel('Avg Rating')
ax.set_xticks(range(0, 24))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/charts/07_temporal_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInsight: Rating cao nhất vào khoảng 2–3h sáng (người dùng ít nhưng có xu hướng rating cẩn thận hơn)')

## B. DEMOGRAPHICS ANALYSIS

### B1. Rating theo Giới tính

In [ ]:
# Merge ratings với users để lấy gender
merged = ratings.merge(users[['userId','gender','age_group','occupation_name']], on='userId')

gender_stats = merged.groupby('gender')['rating'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['coral', 'steelblue']
bars = ax.bar(gender_stats['gender'], gender_stats['mean'], color=colors, edgecolor='black')
ax.set_title('Rating trung bình theo giới tính')
ax.set_ylabel('Avg Rating')
ax.set_ylim(3.5, 4.0)
for bar, val in zip(bars, gender_stats['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/charts/07_demographics_gender.png', dpi=150, bbox_inches='tight')
plt.show()

### B2. Rating theo Nhóm tuổi

In [ ]:
age_order = ['Under 18','18-24','25-34','35-44','45-49','50-55','56+']
age_stats = merged.groupby('age_group')['rating'].agg(['mean', 'count']).reset_index()
age_stats['age_order'] = age_stats['age_group'].map({v:i for i,v in enumerate(age_order)})
age_stats = age_stats.sort_values('age_order')

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(age_stats['age_group'], age_stats['mean'], color='steelblue', edgecolor='black')
ax.set_title('Rating trung bình theo nhóm tuổi')
ax.set_xlabel('Nhóm tuổi')
ax.set_ylabel('Avg Rating')
ax.set_xticklabels(age_stats['age_group'], rotation=15, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/charts/07_demographics_age.png', dpi=150, bbox_inches='tight')
plt.show()

### B3. Genre Preferences theo Giới tính

In [ ]:
# Thêm genre flags
all_genres = [
    'Action','Adventure','Animation',"Children's",'Comedy','Crime',
    'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical',
    'Mystery','Romance','Sci-Fi','Thriller','War','Western'
]

for genre in all_genres:
    movies[f'is_{genre}'] = movies['genres'].str.contains(genre, na=False).astype(int)

# Merge với ratings
merged_movies = merged.merge(movies[['movieId'] + [f'is_{g}' for g in all_genres]], on='movieId')

# Tính avg rating theo genre cho M và F
results = []
for genre in all_genres:
    col = f'is_{genre}'
    male = merged_movies[(merged_movies['gender']=='M') & (merged_movies[col]==1)]['rating'].mean()
    female = merged_movies[(merged_movies['gender']=='F') & (merged_movies[col]==1)]['rating'].mean()
    results.append({
        'genre': genre,
        'male_avg': male,
        'female_avg': female,
        'diff': female - male
    })

genre_gender = pd.DataFrame(results).sort_values('diff', ascending=False)
print(genre_gender)

In [ ]:
# Vẽ so sánh genre preference
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(genre_gender))
width = 0.35

bars1 = ax.bar(x - width/2, genre_gender['male_avg'], width, label='Male', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, genre_gender['female_avg'], width, label='Female', color='coral', edgecolor='black')

ax.set_xlabel('Genre')
ax.set_ylabel('Avg Rating')
ax.set_title('Genre Preferences: Male vs Female')
ax.set_xticks(x)
ax.set_xticklabels(genre_gender['genre'], rotation=30, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(3.0, 4.2)

plt.tight_layout()
plt.savefig('results/charts/07_genre_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()

### B4. Genre Preferences theo Nhóm tuổi

In [ ]:
# Top genres cho mỗi nhóm tuổi
top_by_age = {}
for age_grp in age_order:
    subset = merged_movies[merged_movies['age_group'] == age_grp]
    genre_avgs = []
    for genre in all_genres:
        col = f'is_{genre}'
        avg = subset[subset[col]==1]['rating'].mean()
        if pd.notna(avg):
            genre_avgs.append({'genre': genre, 'avg': avg})
    top = sorted(genre_avgs, key=lambda x: x['avg'], reverse=True)[:5]
    top_by_age[age_grp] = top

# Vẽ heatmap
heatmap_data = pd.DataFrame(top_by_age).T
fig, ax = plt.subplots(figsize=(14, 6))

# Chuyển thành ma trận
hm = np.zeros((len(age_order), len(all_genres)))
for i, age_grp in enumerate(age_order):
    subset = merged_movies[merged_movies['age_group'] == age_grp]
    for j, genre in enumerate(all_genres):
        col = f'is_{genre}'
        hm[i, j] = subset[subset[col]==1]['rating'].mean()

im = ax.imshow(hm, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(all_genres)))
ax.set_xticklabels(all_genres, rotation=35, ha='right')
ax.set_yticks(range(len(age_order)))
ax.set_yticklabels(age_order)
ax.set_title('Genre Preferences by Age Group (Avg Rating)')
plt.colorbar(im, ax=ax, label='Avg Rating')

plt.tight_layout()
plt.savefig('results/charts/07_genre_by_age_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tổng kết Context Features

**Temporal Insights:**
- Rating có xu hướng giảm nhẹ qua các năm (1999–2000)
- Cuối tuần (Sat/Sun) có rating trung bình cao hơn WFH ngày thường
- Giờ sáng sớm (2–6h) có rating cao nhất

**Demographics Insights:**
- Male: Thích Action, Sci-Fi, Thriller hơn
- Female: Thích Romance, Drama, Comedy hơn
- Nhóm 18-24 tuổi: Đánh giá phim cao nhất (hive style)
- Trên 56+: Thích Drama, Film-Noir hơn

**Ứng dụng:** Context features có thể cải thiện recommendation bằng cách:
- Context-aware CF: weight ratings theo thời gian gần hơn
- Demographic-aware: bias gợi ý theo nhóm đối tượng